# Regression Discontinuity Design: Concepts

In many settings, treatment is assigned based on whether a **running variable** crosses a **cutoff threshold**. For example:

- Students scoring above 90 get a scholarship
- Districts with population above 10,000 receive extra funding
- Candidates winning by any margin take office

**Regression Discontinuity Design (RDD)** exploits this sharp assignment rule to estimate causal effects. The core idea: units just below and just above the cutoff are nearly identical in all respects *except* treatment status. Any jump in the outcome at the cutoff is attributable to the treatment.

Formally, if treatment $D_i = \mathbf{1}(X_i \geq c)$ for running variable $X_i$ and cutoff $c$, then the **local average treatment effect at the cutoff** is:

$$\tau_{\text{RDD}} = \lim_{x \downarrow c} E[Y_i | X_i = x] \;-\; \lim_{x \uparrow c} E[Y_i | X_i = x]$$

This is the difference in conditional expectation functions (CEFs) approaching the cutoff from each side.

**Key assumption:** potential outcomes $Y(0)$ and $Y(1)$ are continuous functions of $X$ at the cutoff. If this holds, any discontinuity in the *observed* outcome at the cutoff must be caused by the treatment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf

# Reproducibility
np.random.seed(42)

# Plotting style
plt.rcParams.update({
    'figure.figsize': (9, 5),
    'font.size': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

## Simulate Sharp RDD Data

We generate a simple sharp RDD scenario:

- **Running variable** $X_i \sim \text{Uniform}(-1, 1)$
- **Cutoff** at $c = 0$
- **Treatment** $D_i = \mathbf{1}(X_i \geq 0)$
- **Outcome** $Y_i = 2 + 1.5 X_i + 3 D_i + \varepsilon_i$, where $\varepsilon_i \sim N(0, 0.5)$

The **true treatment effect is $\tau = 3$**. Our goal is to recover this from the data.

In [ ]:
n = 500
cutoff = 0
true_effect = 3

# Running variable: uniform on [-1, 1]
X = np.random.uniform(-1, 1, n)

# Sharp treatment assignment at the cutoff
D = (X >= cutoff).astype(int)

# Outcome with linear relationship + treatment jump + noise
epsilon = np.random.normal(0, 0.5, n)
Y = 2 + 1.5 * X + true_effect * D + epsilon

df = pd.DataFrame({'X': X, 'D': D, 'Y': Y})
print(f"n = {n}, treated = {D.sum()}, control = {n - D.sum()}")
df.head(10)

## Visualize the Discontinuity

The signature of an RDD is a **visible jump** in the outcome at the cutoff. On each side, the relationship between $X$ and $Y$ is smooth; the only discontinuity is the treatment effect.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Scatter: control (blue) and treated (orange)
control = df[df['D'] == 0]
treated = df[df['D'] == 1]
ax.scatter(control['X'], control['Y'], alpha=0.4, s=20, color='steelblue', label='Control (X < 0)')
ax.scatter(treated['X'], treated['Y'], alpha=0.4, s=20, color='darkorange', label='Treated (X >= 0)')

# Fitted lines on each side
for subset, color in [(control, 'steelblue'), (treated, 'darkorange')]:
    slope, intercept, _, _, _ = stats.linregress(subset['X'], subset['Y'])
    x_line = np.linspace(subset['X'].min(), subset['X'].max(), 100)
    ax.plot(x_line, intercept + slope * x_line, color=color, linewidth=2.5)

# Vertical line at cutoff
ax.axvline(x=cutoff, color='gray', linestyle='--', linewidth=1, alpha=0.7)

# Annotate the gap
# Predicted values at cutoff from each side
slope_c, int_c, _, _, _ = stats.linregress(control['X'], control['Y'])
slope_t, int_t, _, _, _ = stats.linregress(treated['X'], treated['Y'])
y_below = int_c + slope_c * cutoff
y_above = int_t + slope_t * cutoff

ax.annotate('', xy=(0.03, y_above), xytext=(0.03, y_below),
            arrowprops=dict(arrowstyle='<->', color='red', lw=2))
ax.text(0.08, (y_above + y_below) / 2, f'Gap $\\approx$ {y_above - y_below:.2f}',
        fontsize=13, color='red', va='center')

ax.set_xlabel('Running Variable (X)')
ax.set_ylabel('Outcome (Y)')
ax.set_title('Sharp RDD: Discontinuity at the Cutoff')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

## Estimate with OLS

Two approaches:

1. **Naive OLS**: regress $Y$ on $D$ alone, ignoring the running variable. This is biased because $X$ affects both treatment assignment and outcomes.

2. **RDD OLS**: regress $Y$ on $X$, $D$, and $X \times D$. This allows different slopes on each side of the cutoff. The coefficient on $D$ gives the treatment effect at the cutoff.

$$Y_i = \alpha + \beta_1 X_i + \tau D_i + \beta_2 (X_i \cdot D_i) + \varepsilon_i$$

Here $\tau$ is our estimate of the treatment effect at $X = 0$.

In [ ]:
# Naive OLS: ignores the running variable
naive = smf.ols('Y ~ D', data=df).fit()
print("=== Naive OLS: Y ~ D ===")
print(f"Estimated treatment effect: {naive.params['D']:.3f}")
print(f"95% CI: [{naive.conf_int().loc['D', 0]:.3f}, {naive.conf_int().loc['D', 1]:.3f}]")
print(f"True effect: {true_effect}")
print(f"Bias: {naive.params['D'] - true_effect:.3f}")
print()

# RDD OLS: includes running variable and interaction
df['XD'] = df['X'] * df['D']
rdd = smf.ols('Y ~ X + D + XD', data=df).fit()
print("=== RDD OLS: Y ~ X + D + X*D ===")
print(f"Estimated treatment effect (coeff on D): {rdd.params['D']:.3f}")
print(f"95% CI: [{rdd.conf_int().loc['D', 0]:.3f}, {rdd.conf_int().loc['D', 1]:.3f}]")
print(f"True effect: {true_effect}")
print(f"Bias: {rdd.params['D'] - true_effect:.3f}")
print()
print(rdd.summary())

## Local Linear Regression

Using all the data assumes the linear model is correct everywhere. In practice, we might worry that the true relationship between $X$ and $Y$ is nonlinear far from the cutoff. 

**Local linear regression** addresses this by restricting estimation to observations within a **bandwidth** $h$ of the cutoff. We fit separate linear regressions on each side using only data where $|X_i| \leq h$.

The tradeoff:
- **Large $h$**: more data, more precise estimates, but potentially biased if the relationship is nonlinear far from the cutoff
- **Small $h$**: less bias (we're truly comparing "near-identical" units), but fewer observations and noisier estimates

In [ ]:
def local_linear_rdd(data, bandwidth):
    """Estimate RDD treatment effect using local linear regression within bandwidth."""
    local = data[data['X'].abs() <= bandwidth].copy()
    local['XD'] = local['X'] * local['D']
    if len(local) < 10:
        return np.nan, np.nan, np.nan, 0
    model = smf.ols('Y ~ X + D + XD', data=local).fit()
    tau = model.params['D']
    ci = model.conf_int().loc['D']
    return tau, ci[0], ci[1], len(local)

# Try three bandwidths
bandwidths = [0.5, 0.3, 0.1]

print(f"{'Bandwidth':>10}  {'Estimate':>10}  {'95% CI':>20}  {'n_local':>8}")
print("-" * 55)
for h in bandwidths:
    tau, lo, hi, n_local = local_linear_rdd(df, h)
    print(f"{h:>10.2f}  {tau:>10.3f}  [{lo:>8.3f}, {hi:>8.3f}]  {n_local:>8d}")
print(f"\nTrue effect: {true_effect}")

In [ ]:
# Visualize the three bandwidths
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, h in zip(axes, bandwidths):
    local = df[df['X'].abs() <= h]
    ctrl = local[local['D'] == 0]
    trt = local[local['D'] == 1]
    
    ax.scatter(ctrl['X'], ctrl['Y'], alpha=0.5, s=15, color='steelblue')
    ax.scatter(trt['X'], trt['Y'], alpha=0.5, s=15, color='darkorange')
    
    # Fit lines on each side
    for subset, color in [(ctrl, 'steelblue'), (trt, 'darkorange')]:
        if len(subset) > 2:
            slope, intercept, _, _, _ = stats.linregress(subset['X'], subset['Y'])
            x_line = np.linspace(subset['X'].min(), subset['X'].max(), 50)
            ax.plot(x_line, intercept + slope * x_line, color=color, linewidth=2)
    
    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    tau, _, _, n_local = local_linear_rdd(df, h)
    ax.set_title(f'h = {h}, n = {n_local}\n$\\hat{{\\tau}}$ = {tau:.2f}')
    ax.set_xlabel('X')

axes[0].set_ylabel('Y')
fig.suptitle('Local Linear Regression at Different Bandwidths', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Bandwidth Sensitivity

A natural question: how sensitive are our estimates to the choice of bandwidth? We sweep over a range of bandwidths and plot the resulting estimates. If the effect is real, estimates should be **stable** across a range of reasonable bandwidths.

In [ ]:
# Sweep over bandwidths
h_grid = np.arange(0.05, 1.01, 0.05)
results = []

for h in h_grid:
    tau, lo, hi, n_local = local_linear_rdd(df, h)
    results.append({'h': h, 'tau': tau, 'ci_lo': lo, 'ci_hi': hi, 'n': n_local})

res = pd.DataFrame(results).dropna()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(res['h'], res['tau'], 'o-', color='steelblue', markersize=5, label='Estimated $\\hat{\\tau}$')
ax.fill_between(res['h'], res['ci_lo'], res['ci_hi'], alpha=0.2, color='steelblue', label='95% CI')
ax.axhline(y=true_effect, color='red', linestyle='--', linewidth=1.5, label=f'True effect = {true_effect}')

ax.set_xlabel('Bandwidth (h)')
ax.set_ylabel('Estimated Treatment Effect')
ax.set_title('RDD Estimate vs. Bandwidth')
ax.legend()
plt.tight_layout()
plt.show()

## McCrary Density Test

RDD assumes that people cannot precisely manipulate the running variable to place themselves on the preferred side of the cutoff. If they can, the "as-if random" logic near the cutoff breaks down.

The **McCrary density test** checks for this by looking at the distribution of the running variable around the cutoff. A discontinuity in the *density* of $X$ at the cutoff is evidence of manipulation.

We'll look at two cases:
1. Our simulated data (uniform $X$, no manipulation)
2. A "manipulated" version where some units just below the cutoff are pushed above it

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel 1: No manipulation (our data)
ax = axes[0]
ax.hist(df['X'], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(x=0, color='red', linestyle='--', linewidth=1.5)
ax.set_xlabel('Running Variable (X)')
ax.set_ylabel('Count')
ax.set_title('No Manipulation\n(Density is smooth at cutoff)')

# Panel 2: Manipulated data
# Take units in [-0.15, 0) and push 70% of them just above the cutoff
X_manip = X.copy()
near_below = (X_manip >= -0.15) & (X_manip < 0)
push_mask = near_below & (np.random.random(n) < 0.7)
X_manip[push_mask] = np.abs(X_manip[push_mask]) + np.random.uniform(0, 0.02, push_mask.sum())

ax = axes[1]
ax.hist(X_manip, bins=40, color='darkorange', edgecolor='white', alpha=0.8)
ax.axvline(x=0, color='red', linestyle='--', linewidth=1.5)
ax.set_xlabel('Running Variable (X)')
ax.set_ylabel('Count')
ax.set_title('With Manipulation\n(Bunching just above cutoff)')

plt.tight_layout()
plt.show()

print("Left panel: uniform distribution, no bunching. RDD assumptions look fine.")
print("Right panel: suspicious spike just above the cutoff. People are gaming the threshold.")

## Discussion: Key Takeaways

**What RDD gives us.** When treatment is assigned at a cutoff, RDD provides a credible causal estimate by comparing outcomes just above and just below the threshold. Units near the cutoff are similar in all observable and unobservable characteristics; the only difference is whether they received treatment.

**The core assumption.** Potential outcomes $Y(0)$ and $Y(1)$ must be *continuous* at the cutoff. If this holds, any discontinuity in the observed outcome is caused by the treatment. This is violated when people can precisely manipulate the running variable (hence the McCrary test).

**The estimate is local.** RDD identifies the treatment effect *at the cutoff*, for the population of units near the threshold. It does not tell us the effect for units far from the cutoff. This is a local average treatment effect (LATE), similar in spirit to what we saw with instrumental variables.

**Bandwidth is a research choice.** Wider bandwidths give more precise estimates but risk bias from misspecification; narrower bandwidths reduce bias but increase variance. The bandwidth sensitivity plot should show stability across a range of reasonable choices. Formal methods exist for optimal bandwidth selection (Imbens and Kalyanaraman 2012; Calonico, Cattaneo, and Titiunik 2014).

**Always check for manipulation.** The McCrary density test (or visual inspection of the running variable's histogram) is a standard diagnostic. If the density is discontinuous at the cutoff, the RDD assumptions are suspect.

**Next class:** we will see RDD applied to a real policy question, using Mo and Conn (2018) on the effect of Teach For America corps members on student achievement, where assignment depends on a test score cutoff.